In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.3 MB/s eta 0:00:00


In [4]:
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found in Colab Secrets. Please add it.")

    print("✅ API Key loaded successfully.")

except Exception as e:
    print(f"❌ Error loading API key: {e}")
    GROQ_API_KEY = None

✅ API Key loaded successfully.


In [5]:
from groq import Groq
import re

client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.1-8b-instant"
TEMPERATURE = 0

In [6]:
from groq import Groq
import re

# Initialize Groq client with the API key loaded from Colab Secrets
# Ensure GROQ_API_KEY is defined from the previous cell.
if 'GROQ_API_KEY' in locals() and GROQ_API_KEY is not None:
    client = Groq(api_key=GROQ_API_KEY)
    print("Groq client initialized with secret API key.")
else:
    print("GROQ_API_KEY not available, Groq client not initialized.")

MODEL = "llama-3.1-8b-instant"
TEMPERATURE = 0

Groq client initialized with secret API key.


In [7]:
def ask_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [8]:
def check_constraints(output):
    violations = []

    # Remove empty lines
    lines = [line.strip() for line in output.split("\n") if line.strip()]

    # Check exactly 3 bullet points
    bullets = [line for line in lines if line.startswith("-")]

    if len(bullets) != 3:
        violations.append(
            f"Expected exactly 3 bullet points, found {len(bullets)}"
        )

    # Check every bullet starts with a verb
    verbs = [
        "Track", "Tracks",
        "Display", "Displays",
        "Monitor", "Monitors",
        "Measure", "Measures",
        "Provide", "Provides",
        "Offer", "Offers",
        "Connect", "Connects",
        "Record", "Records"
    ]

    for bullet in bullets:
        text = bullet[1:].strip()
        if not any(text.startswith(v) for v in verbs):
            violations.append(
                f"Bullet does not start with a verb: {bullet}"
            )

    # Check banned word
    if "smart" in output.lower():
        violations.append(
            "Used prohibited word 'smart'"
        )

    # Check word count
    if len(output.split()) > 60:
        violations.append(
            "Description exceeds 60 words"
        )

    return violations

In [9]:
initial_prompt = """
Describe a smartwatch.

Constraints:
1. Use exactly 3 bullet points.
2. Each bullet point must start with a verb.
3. Do not use the word "smart" anywhere.

Keep the description under 60 words.
"""

In [10]:
initial_response = ask_llm(initial_prompt)

print("=== INITIAL PROMPT ===")
print(initial_prompt)

print("\nResponse:")
print(initial_response)

violations = check_constraints(initial_response)

print("\nViolations:")
if violations:
    for v in violations:
        print("-", v)
else:
    print("No violations")

=== INITIAL PROMPT ===

Describe a smartwatch.

Constraints:
1. Use exactly 3 bullet points.
2. Each bullet point must start with a verb.
3. Do not use the word "smart" anywhere.

Keep the description under 60 words.


Response:
This wearable device tracks various aspects of a user's life. 
• Displaying vital signs such as heart rate and blood pressure.
• Receiving notifications from connected phones and other devices.
• Allowing users to control music playback and access various apps.

Violations:
- Expected exactly 3 bullet points, found 0


In [11]:
refined_prompt = """
Describe a fitness watch.

Follow these rules strictly:

- Write exactly 3 bullet points.
- Every bullet point must begin with an action verb.
- Never use the word "smart".
- Keep the complete description below 60 words.

Example format:
- Tracks health activities.
- Displays daily progress.
- Monitors fitness goals.

Do not add any introduction or conclusion.
Only provide the three bullet points.
"""

In [12]:
refined_response = ask_llm(refined_prompt)

print("\n\n=== REFINED PROMPT ===")
print(refined_prompt)

print("\nResponse:")
print(refined_response)

refined_violations = check_constraints(refined_response)

print("\nViolations:")
if refined_violations:
    for v in refined_violations:
        print("-", v)
else:
    print("No violations")



=== REFINED PROMPT ===

Describe a fitness watch.

Follow these rules strictly:

- Write exactly 3 bullet points.
- Every bullet point must begin with an action verb.
- Never use the word "smart".
- Keep the complete description below 60 words.

Example format:
- Tracks health activities.
- Displays daily progress.
- Monitors fitness goals.

Do not add any introduction or conclusion.
Only provide the three bullet points.


Response:
- Tracks health activities.
- Displays daily progress.
- Monitors fitness goals.

Violations:
No violations


In [13]:
print("\n=== COMPARISON ===")

print("\nInitial Violations:")
print(len(violations))

print("\nRefined Violations:")
print(len(refined_violations))

if len(refined_violations) < len(violations):
    print("\nImprovement achieved after refinement.")
else:
    print("\nNo improvement detected.")


=== COMPARISON ===

Initial Violations:
1

Refined Violations:
0

Improvement achieved after refinement.
